(to be deleted before merging)

(this is just converting some of our old data sources to the new format for dev/testing/demo)

In [1]:
import pandas as pd
import pandera as pa

In [2]:
from wholecell.io.schemas import RnaseqSamplesManifestSchema, RnaseqTpmTableSchema


In [3]:
ref_data = pd.read_csv('../../reconstruction/ecoli/flat/rna_seq_data/rnaseq_rsem_tpm_mean.tsv', sep='\t').rename(
    columns={'Gene': 'gene_id'}
)
ref_data.head()

,gene_id,M9 Glucose minus AAs,N limited Glucose minus AAs,M9 Glucose plus AAs,M9 Glycerol plus AAs,P limited Glucose plus AAs
0,EG10001,58.686667,71.140000,81.363333,45.183333,45.710000
1,EG10002,115.996667,81.043333,213.696667,421.233333,410.366667
2,EG10003,48.106667,46.103333,37.333333,47.300000,41.936667
3,EG10004,133.000000,133.340000,110.883333,126.246667,75.233333
4,EG10006,1.553333,2.190000,12.140000,2.046667,2.103333


In [4]:
ref_data.sum()

gene_id                        EG10001EG10002EG10003EG10004EG10006EG10007EG10...
M9 Glucose minus AAs                                               999999.896667
N limited Glucose minus AAs                                        999999.913333
M9 Glucose plus AAs                                                999999.996667
M9 Glycerol plus AAs                                              1000000.056667
P limited Glucose plus AAs                                             999999.96
dtype: object

In [5]:
ginkgo_data = pd.read_csv('rnaseq_exp96546_tpms_mean.tsv', sep='\t')
ginkgo_data.head()

,gene_id,11_J3_Modified_M9_N_Fe,MG1655 rph+ ?trpR ?tnaA [+cymR +vio_J3 +vioE]_Modified_M9_N_Fe_5gperL_Trp,MG1655 rph+ ?trpR ?tnaA [+cymR +vio_J3]_Modified_M9_N_Fe_5gperL_Trp,MG1655 rph+ ?trpR ?tnaA [+cymR +vio_M5]_Modified_M9_N_Fe,MG1655 rph+ ?trpR ?tnaA [+cymR +vio_M5]_Modified_M9_N_Fe_5gperL_Trp,MG1655 rph+ [+cymR +vio_J3]_Modified_M9_N_Fe,MG1655 rph+ [+cymR +vio_M5]_Modified_M9_N_Fe,MG1655 rph+_EZ_Rich,MG1655 rph+_Modified_M9_N_Fe
0,EG10001,55.458648,74.029122,42.569137,71.754026,69.918466,84.082632,65.935780,27.705990,77.887282
1,EG10002,13.607349,130.361037,42.569229,102.708732,57.242007,85.216886,157.612550,53.307287,253.574400
2,EG10003,21.396915,21.968495,11.310599,21.073006,20.846303,30.207164,18.664348,10.216092,23.208875
3,EG10004,40.406313,54.862116,32.328811,51.996717,57.950094,69.973328,44.089640,19.530986,59.500494
4,EG10006,48.062423,14.585427,2.695886,32.389446,13.733274,10.482831,36.929228,4.261452,70.181742


In [6]:
ginkgo_data.sum()

gene_id                                                                      EG10001EG10002EG10003EG10004EG10006EG10007EG10...
11_J3_Modified_M9_N_Fe                                                                                           974227.292631
MG1655 rph+ ?trpR ?tnaA [+cymR +vio_J3 +vioE]_Modified_M9_N_Fe_5gperL_Trp                                        997546.318216
MG1655 rph+ ?trpR ?tnaA [+cymR +vio_J3]_Modified_M9_N_Fe_5gperL_Trp                                              997137.605894
MG1655 rph+ ?trpR ?tnaA [+cymR +vio_M5]_Modified_M9_N_Fe                                                         988485.587402
MG1655 rph+ ?trpR ?tnaA [+cymR +vio_M5]_Modified_M9_N_Fe_5gperL_Trp                                              988973.172898
MG1655 rph+ [+cymR +vio_J3]_Modified_M9_N_Fe                                                                     997987.409079
MG1655 rph+ [+cymR +vio_M5]_Modified_M9_N_Fe                                                                   

In [7]:
RnaseqSamplesManifestSchema.columns

{'dataset_id': <Schema Column(name=dataset_id, type=DataType(str))>,
 'dataset_description': <Schema Column(name=dataset_description, type=DataType(str))>,
 'file_path': <Schema Column(name=file_path, type=DataType(str))>,
 'data_source': <Schema Column(name=data_source, type=DataType(str))>,
 'data_source_experiment_id': <Schema Column(name=data_source_experiment_id, type=DataType(str))>,
 'data_source_date': <Schema Column(name=data_source_date, type=DataType(str))>,
 'strain': <Schema Column(name=strain, type=DataType(str))>,
 'condition': <Schema Column(name=condition, type=DataType(str))>}

In [8]:
manifest = pd.DataFrame(columns=RnaseqSamplesManifestSchema.columns)
manifest

,dataset_id,dataset_description,file_path,data_source,data_source_experiment_id,data_source_date,strain,condition


In [9]:
def generate_tpm_table(
    original_tpm_table,
    original_column_name,
    manifest,
    out_file_path,
    dataset_id,
    dataset_description,
    data_source,
    data_source_experiment_id,
    data_source_date,
    strain,
    condition,  
):
    # Load the data
    tpm_table = original_tpm_table[['gene_id',original_column_name]].rename(columns={original_column_name: 'tpm_mean'})
    # Create a dictionary for the new row
    new_row = {
        "dataset_id": dataset_id,
        "dataset_description": dataset_description,
        "file_path": dataset_id+'.tsv',
        "data_source": data_source,
        "data_source_experiment_id": data_source_experiment_id,
        "data_source_date": data_source_date,
        "strain": strain,
        "condition": condition,
    }
    # Append new row to manifest DataFrame
    manifest.loc[len(manifest)] = new_row

    tpm_table.to_csv(out_file_path+dataset_id+'.tsv', sep='\t', index=False)
    
    return manifest, tpm_table


In [10]:
manifest, ref_m9glc = generate_tpm_table(
    original_tpm_table=ref_data,
    original_column_name='M9 Glucose minus AAs',
    manifest=manifest,
    out_file_path='../',
    dataset_id='ref_0001',
    dataset_description='reference data (reconstruction/flat/rna_seq_data/rnaseq_rsem_tpm_mean.tsv), M9 Glucose minus AAs',
    data_source='reconstruction/flat/rna_seq_data/rnaseq_rsem_tpm_mean.tsv',
    data_source_experiment_id='n/a',
    data_source_date='n/a',
    strain='n/a',
    condition='M9, Glucose, Aerobic, 37C'
)

In [11]:
manifest, ref_m9glc = generate_tpm_table(
    original_tpm_table=ref_data,
    original_column_name='M9 Glucose plus AAs',
    manifest=manifest,
    out_file_path='../',
    dataset_id='ref_0002',
    dataset_description='reference data (reconstruction/flat/rna_seq_data/rnaseq_rsem_tpm_mean.tsv), M9 Glucose plus AAs',
    data_source='reconstruction/flat/rna_seq_data/rnaseq_rsem_tpm_mean.tsv',
    data_source_experiment_id='n/a',
    data_source_date='n/a',
    strain='n/a',
    condition='M9, Glucose+AAs, Aerobic, 37C'
)

In [12]:
manifest, mg1655_tpm_table = generate_tpm_table(
    original_tpm_table=ginkgo_data,
    original_column_name='MG1655 rph+_Modified_M9_N_Fe',
    manifest=manifest,
    out_file_path='../',
    dataset_id='gbw_0001',
    dataset_description='MG1655 rph+_Modified_M9_N_Fe, average of 3h and 4h timepoints',
    data_source='Ginkgo Bioworks',
    data_source_experiment_id='exp96546',
    data_source_date='2025-11-24',
    strain='MG1655 rph+',
    condition='Modified_M9_N_Fe, Glucose, Aerobic, 37C',
)

In [13]:
manifest, mg1655_ezrich_tpm_table = generate_tpm_table(
    original_tpm_table=ginkgo_data,
    original_column_name='MG1655 rph+_EZ_Rich',
    manifest=manifest,
    out_file_path='../',
    dataset_id='gbw_0002',
    dataset_description='MG1655 rph+, EZ_Rich, average of 3h and 4h timepoints',
    data_source='Ginkgo Bioworks',
    data_source_experiment_id='exp96546',
    data_source_date='2025-11-24',
    strain='MG1655 rph+',
    condition='Modified_M9_N_Fe, EZ Rich media, Aerobic, 37C',
)

In [14]:
# manifest, M5_tpm_table = generate_tpm_table(
#     original_column_name='MG1655 rph+ ?trpR ?tnaA [+cymR +vio_M5]_Modified_M9_N_Fe',
#     manifest=manifest,
#     dataset_id='GBW_exp96546__MG1655_rph+_delTRPR_delTNAA_cymR_vioM5__Modified_M9_N_Fe',
#     dataset_description='MG1655 rph+ del(trpR,tnaA) cymR+vio_M5, average of 3h and 4h timepoints',
#     file_path='GBW_exp96546__MG1655_rph+_delTRPR_delTNAA_cymR_vioM5__Modified_M9_N_Fe',
#     data_source='Ginkgo Bioworks',
#     data_source_experiment_id='exp96546',
#     data_source_date='2025-11-24',
#     strain='MG1655 rph+ del(trpR,tnaA) cymR+vio_M5',
#     condition='Modified_M9_N_Fe, Glucose, Aerobic, 37C',
# )

In [15]:
manifest

,dataset_id,dataset_description,file_path,data_source,data_source_experiment_id,data_source_date,strain,condition
0,ref_0001,reference data (reconstruction/flat/rna_seq_da...,ref_0001.tsv,reconstruction/flat/rna_seq_data/rnaseq_rsem_t...,n/a,n/a,n/a,"M9, Glucose, Aerobic, 37C"
1,ref_0002,reference data (reconstruction/flat/rna_seq_da...,ref_0002.tsv,reconstruction/flat/rna_seq_data/rnaseq_rsem_t...,n/a,n/a,n/a,"M9, Glucose+AAs, Aerobic, 37C"
2,gbw_0001,"MG1655 rph+_Modified_M9_N_Fe, average of 3h an...",gbw_0001.tsv,Ginkgo Bioworks,exp96546,2025-11-24,MG1655 rph+,"Modified_M9_N_Fe, Glucose, Aerobic, 37C"
3,gbw_0002,"MG1655 rph+, EZ_Rich, average of 3h and 4h tim...",gbw_0002.tsv,Ginkgo Bioworks,exp96546,2025-11-24,MG1655 rph+,"Modified_M9_N_Fe, EZ Rich media, Aerobic, 37C"


In [16]:
manifest.to_csv('../manifest.tsv', sep='\t', index=False)